<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Nano Transfer with Cosmos Framework

This notebook runs Cosmos3-Nano **video transfer** inference through the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

Transfer generates a target clip from a caption (`prompt.json`) and a spatial control video on the hint block (`control_path`). Supported cookbook controls:

- **edge** — Canny edge map (`control_edge.mp4`)
- **blur** — blurred reference (`control_blur.mp4`)
- **depth** — depth map (`control_depth.mp4`)
- **seg** — segmentation map (`control_seg.mp4`)
- **wsm** — world-scenario map (`control_wsm.mp4`, 101 frames @ 10 FPS)

vLLM-Omni does not expose transfer controls today; use this Cosmos Framework path only.

Sections **8–12** each run one control (inference + preview). Run only the blocks you need.

> **GPU required.** Run on a host where §3 (`nvidia-smi`) and §7 (`cuda available: True`) both pass.

**Self-contained setup:** everything needed to run this notebook (system packages, clone, Python venv) is in §2–§7 below — no external bootstrap scripts required.

Workflow: §2 configure → §3 GPU check → §4 system packages → §5 clone framework → §6 install → §7 verify → §8 review specs → §9–§13 inference and preview (run only the controls you need).


## 1. Prerequisites

1. Linux with NVIDIA GPU access (`nvidia-smi` visible where you run this notebook).
2. `git` (§5 clones Cosmos Framework; §4 installs `git-lfs` when `apt-get` is available).
3. Outbound network for `git clone`, PyPI (`uv sync` in §6), and Hugging Face checkpoints (`HF_TOKEN` or `uvx hf auth login`).
4. Sample inputs under [`assets/`](./assets) and specs under [`specs/`](./specs) (shipped with this cookbook).
5. §2 picks `COSMOS3_UV_GROUP` from your CPU/arch: `cu130-train` for CUDA 13 or `aarch64`; `cu128-train` for CUDA 12 on x86_64 (override if your driver does not match).

§4–§6 install system libraries, clone the framework when missing, install [`uv`](https://docs.astral.sh/uv/) if needed, run `uv sync`, and create `packages/cosmos3/.venv`. Caches default to `generator/transfer/.cache/`.


## 2. Configure Paths

Defaults assume this `cosmos` checkout with the framework at `packages/cosmos3`. Override in the next cell or via the environment variables below.

```bash
export COSMOS3_REPO=/path/to/cosmos-framework   # or packages/cosmos3 in this repo
export COSMOS3_UV_GROUP=cu130-train               # cu128-train on x86 + CUDA 12.x; §2 picks aarch64 defaults
export COSMOS3_CACHE_ROOT=/path/to/cache          # optional; else ~/.cache/{uv,huggingface}
export COSMOS3_TRANSFER_OUTPUT_ROOT=/path/to/outputs
export CUDA_VISIBLE_DEVICES=0
```


## 3. Confirm GPU access

Run this **before** install (§6) or inference (§9+). If it fails, fix GPU allocation or driver setup before continuing.

In [ ]:
from pathlib import Path
import json
import os
import platform
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos3"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_TRANSFER_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "transfer"
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get(
    "COSMOS3_GIT_URL",
    "https://github.com/NVIDIA/cosmos-framework.git",
)
def default_uv_group() -> str:
    # cu128-train attention wheels are x86_64-only; aarch64 hosts need cu130-train for natten.
    if platform.machine() == "aarch64":
        return "cu130-train"
    return "cu128-train"


COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", default_uv_group())
COSMOS3_TRANSFER_OUTPUT_ROOT = Path(
    os.environ.get(
        "COSMOS3_TRANSFER_OUTPUT_ROOT",
        COSMOS3_TRANSFER_ROOT / "outputs" / "notebooks",
    )
).resolve()
COSMOS3_SPECS_DIR = COSMOS3_TRANSFER_ROOT / "specs"
TRANSFER_CONTROLS = ("edge", "blur", "depth", "seg", "wsm")

os.environ["COSMOS_ROOT"] = str(COSMOS_ROOT)
os.environ["COSMOS3_TRANSFER_ROOT"] = str(COSMOS3_TRANSFER_ROOT)
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_TRANSFER_OUTPUT_ROOT"] = str(COSMOS3_TRANSFER_OUTPUT_ROOT)
os.environ.setdefault("COSMOS3_CHECKPOINT_PATH", "Cosmos3-Nano")


def default_cache_path(name: str) -> str:
    root = os.environ.get("COSMOS3_CACHE_ROOT")
    if root:
        return str((Path(root).expanduser() / name).resolve())
    return str((COSMOS3_TRANSFER_ROOT / ".cache" / name).resolve())


os.environ["UV_CACHE_DIR"] = os.environ.get("COSMOS3_UV_CACHE_DIR", default_cache_path("uv"))
os.environ["HF_HOME"] = os.environ.get("COSMOS3_HF_HOME", default_cache_path("huggingface"))
# NGC PyTorch images: clear bundled libtorch from LD_LIBRARY_PATH before inference.
os.environ.pop("LD_LIBRARY_PATH", None)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_MASTER_PORT", free_local_port())

print("cosmos root:", COSMOS_ROOT)
print("transfer cookbook:", COSMOS3_TRANSFER_ROOT)
print("framework:", COSMOS3_REPO)
print("controls:", ", ".join(TRANSFER_CONTROLS))
print("output root:", COSMOS3_TRANSFER_OUTPUT_ROOT)
print("checkpoint:", os.environ["COSMOS3_CHECKPOINT_PATH"])
print("UV_CACHE_DIR:", os.environ["UV_CACHE_DIR"])
print("HF_HOME:", os.environ["HF_HOME"])
print("COSMOS3_UV_GROUP:", os.environ["COSMOS3_UV_GROUP"])
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


In [ ]:
%%bash
set -euo pipefail
echo "hostname: $(hostname)"
echo "CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-<unset>}"
if ! command -v nvidia-smi >/dev/null 2>&1; then
  echo "ERROR: nvidia-smi not found. Run on a GPU host (see §1)."
  exit 1
fi
nvidia-smi -L
GPU_COUNT=$(nvidia-smi -L 2>/dev/null | wc -l)
if [ "${GPU_COUNT:-0}" -lt 1 ]; then
  echo "ERROR: no GPUs visible. Allocate a GPU or set CUDA_VISIBLE_DEVICES."
  exit 1
fi
echo "OK: ${GPU_COUNT} GPU(s) visible on $(hostname)"

## 4. Install system packages (Linux)

Framework guardrails and previews need **ffmpeg**, **git-lfs**, and graphics libraries (`libxcb1`, `libgl1`, …). On hosts with `apt-get` (NGC PyTorch container, many training images), run the next cell to install them.

If `apt-get` is unavailable, install the same packages with your OS package manager — see [Cosmos3 cookbooks README — System packages](../../README.md#system-packages-required-for-framework-guardrails).


In [ ]:
%%bash
set -euo pipefail

PACKAGES=(curl ffmpeg git-lfs libgl1 libglib2.0-0 libx11-dev libxcb1 tree wget)

if command -v apt-get >/dev/null 2>&1; then
  export DEBIAN_FRONTEND=noninteractive
  echo "Installing system packages via apt-get..."
  apt-get update -qq
  apt-get install -y --no-install-recommends "${PACKAGES[@]}"
  echo "OK: apt packages installed"
else
  echo "NOTE: apt-get not found on this host."
  echo "If guardrails or OpenCV fail with libxcb.so.1, install manually:"
  echo "  ${PACKAGES[*]}"
  echo "See cookbooks/cosmos3/README.md (Framework guardrails)."
fi

for cmd in git ffmpeg; do
  command -v "$cmd" >/dev/null || { echo "ERROR: missing $cmd"; exit 1; }
done
if command -v git-lfs >/dev/null 2>&1; then
  git lfs install --skip-repo 2>/dev/null || true
  echo "git-lfs: OK"
else
  echo "WARN: git-lfs not in PATH (uv sync may still work with GIT_LFS_SKIP_SMUDGE=1)"
fi
echo "System package check complete."

## 5. Clone Cosmos Framework

Clones `COSMOS3_GIT_URL` into `COSMOS3_REPO` when the tree is not already present.

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -d "$COSMOS3_REPO/.git" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework tree (no .git): $COSMOS3_REPO"
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
if [ -d .git ]; then
  git status --short --branch
  git remote -v
fi


## 6. Install Cosmos Framework Dependencies

Installs [`uv`](https://docs.astral.sh/uv/) if missing, then runs `uv sync` to create `packages/cosmos3/.venv`. Uses `COSMOS3_UV_GROUP` from §2.

**Skip:** if `.venv` already imports `cosmos_framework` with CUDA available, the next cell skips `uv sync` (fast re-runs). Set `COSMOS3_FORCE_UV_SYNC=1` to force a full re-sync.

For `jupyter execute` on a GPU node, set `COSMOS3_UV_CACHE_DIR` / `COSMOS3_HF_HOME` (or `COSMOS3_CACHE_ROOT`) in §2 first.

If you change `COSMOS3_UV_GROUP`, **re-run this cell** before inference.


In [ ]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH

if ! command -v uv >/dev/null 2>&1; then
  echo "Installing uv..."
  curl -LsSf https://astral.sh/uv/install.sh | sh
  # shellcheck disable=SC1091
  source "${HOME}/.local/bin/env" 2>/dev/null || export PATH="${HOME}/.local/bin:${PATH}"
fi
uv self update 2>/dev/null || true
uv --version

export GIT_LFS_SKIP_SMUDGE=1
mkdir -p "$UV_CACHE_DIR" "$HF_HOME"
cd "$COSMOS3_REPO"
export UV_CACHE_DIR="${UV_CACHE_DIR:?set paths in §2 (run the configure cell first)}"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_REPO/.venv}"
export UV_HTTP_TIMEOUT="${UV_HTTP_TIMEOUT:-600}"
echo "UV_CACHE_DIR=$UV_CACHE_DIR"
echo "COSMOS3_UV_GROUP=$COSMOS3_UV_GROUP"
echo "UV_HTTP_TIMEOUT=$UV_HTTP_TIMEOUT"

if [ -z "${COSMOS3_FORCE_UV_SYNC:-}" ] && [ -x ".venv/bin/python" ]; then
  if env -u LD_LIBRARY_PATH .venv/bin/python -c \
      'import cosmos_framework, torch; assert torch.cuda.is_available()'; then
    echo "venv ready at $COSMOS3_REPO/.venv — skipping uv sync (set COSMOS3_FORCE_UV_SYNC=1 to re-sync)"
    uv pip install imageio imageio-ffmpeg
    env -u LD_LIBRARY_PATH .venv/bin/python -c \
      'import cosmos_framework, torch; print("venv OK (skipped sync)")'
    exit 0
  fi
  echo "Existing .venv failed CUDA/framework check — running full uv sync..."
fi

attempt=1
max_attempts=3
until uv sync --all-extras --group="$COSMOS3_UV_GROUP"; do
  if [ "$attempt" -ge "$max_attempts" ]; then
    echo "ERROR: uv sync failed after $max_attempts attempts (PyPI timeout or network)."
    exit 1
  fi
  echo "uv sync attempt $attempt failed; retrying in 30s..."
  attempt=$((attempt + 1))
  sleep 30
done

uv pip install imageio imageio-ffmpeg

env -u LD_LIBRARY_PATH .venv/bin/python -c \
  'import cosmos_framework, torch; assert torch.cuda.is_available(); print("venv OK")'
echo "Install complete: $COSMOS3_REPO/.venv"


## 7. Verify GPU Environment


In [ ]:
import subprocess

verify_code = r'''
import sys
import torch
print("uv group (env):", __import__("os").environ.get("COSMOS3_UV_GROUP", "?"))
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))
else:
    print("FIX: set COSMOS3_UV_GROUP in §2 (cu130-train or cu128-train), re-run §6 install, then this cell.")
    sys.exit(1)
'''
result = subprocess.run(
    [str(COSMOS3_REPO / ".venv" / "bin" / "python"), "-c", verify_code],
    cwd=str(COSMOS3_REPO),
    env=os.environ.copy(),
)
if result.returncode != 0:
    raise RuntimeError(
        "CUDA not available. Pass §3 first, then re-run §6 install with the correct COSMOS3_UV_GROUP."
    )


## 8. Input Specs and Preview Helpers

Checked-in [`specs/<control>.json`](./specs) use paths relative to `specs/`. Previews use [`preview_helpers.py`](./preview_helpers.py) and `imageio-ffmpeg` installed in §6.

Inference (§9–§13) writes videos to:

```text
<COSMOS3_TRANSFER_OUTPUT_ROOT>/<control>/transfer_<control>/vision.mp4
```


In [ ]:
missing = [c for c in TRANSFER_CONTROLS if not (COSMOS3_SPECS_DIR / f"{c}.json").is_file()]
if missing:
    raise FileNotFoundError(f"missing checked-in specs for {missing} under {COSMOS3_SPECS_DIR}")
print("Using specs:", ", ".join(f"{c}.json" for c in TRANSFER_CONTROLS))


In [ ]:
from preview_helpers import load_transfer_spec, resolve_spec_path

for control in TRANSFER_CONTROLS:
    spec = load_transfer_spec(control)
    block = spec[control]
    print(
        f"{control}: frames={spec.get('num_frames')} fps={spec.get('fps')} "
        f"guidance={spec.get('guidance')} control_guidance={spec.get('control_guidance')} "
        f"control={resolve_spec_path(block['control_path'])}"
    )


## 9. Edge (Canny) Transfer

Run after §7 reports `cuda available: True`.

Precomputed edge control (`control_edge.mp4`) + caption. Output:

```text
<COSMOS3_TRANSFER_OUTPUT_ROOT>/edge/transfer_edge/vision.mp4
```


In [ ]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH
CONTROL=edge
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${CONTROL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL spec=$SPEC output=$OUT_DIR checkpoint=$COSMOS3_CHECKPOINT_PATH"
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --no-guardrails \
  --no-use-torch-compile \
  -i "$SPEC" \
  -o "$OUT_DIR" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2025


### Preview edge


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("edge")


## 10. Blur Transfer

Blurred-reference control (`control_blur.mp4`) + caption. Output: `.../blur/transfer_blur/vision.mp4`.


In [ ]:
%%bash
set -euo pipefail
CONTROL=blur
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${CONTROL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL spec=$SPEC output=$OUT_DIR checkpoint=$COSMOS3_CHECKPOINT_PATH"
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --no-guardrails \
  --no-use-torch-compile \
  -i "$SPEC" \
  -o "$OUT_DIR" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2025


### Preview blur


In [ ]:
from preview_helpers import preview_transfer

preview_transfer("blur")


## 11. Depth Transfer

Depth-map control (`control_depth.mp4`) + caption. Output: `.../depth/transfer_depth/vision.mp4`.


In [ ]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH
CONTROL=depth
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${CONTROL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL spec=$SPEC output=$OUT_DIR checkpoint=$COSMOS3_CHECKPOINT_PATH"
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --no-guardrails \
  --no-use-torch-compile \
  -i "$SPEC" \
  -o "$OUT_DIR" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2025


### Preview depth


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("depth")


## 12. Segmentation Transfer

Segmentation-map control (`control_seg.mp4`) + caption. Output: `.../seg/transfer_seg/vision.mp4`.


In [ ]:
%%bash
set -euo pipefail
CONTROL=seg
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${CONTROL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL spec=$SPEC output=$OUT_DIR checkpoint=$COSMOS3_CHECKPOINT_PATH"
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --no-guardrails \
  --no-use-torch-compile \
  -i "$SPEC" \
  -o "$OUT_DIR" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2025


### Preview seg


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("seg")


## 13. World Scenario (WSM) Transfer

World-scenario control (`control_wsm.mp4`, 101 frames @ 10 FPS) + caption. Output: `.../wsm/transfer_wsm/vision.mp4`.


In [ ]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH
CONTROL=wsm
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${CONTROL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL spec=$SPEC output=$OUT_DIR checkpoint=$COSMOS3_CHECKPOINT_PATH"
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --no-guardrails \
  --no-use-torch-compile \
  -i "$SPEC" \
  -o "$OUT_DIR" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2025


### Preview wsm


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("wsm")
